<a href="https://colab.research.google.com/github/bradleywjenks/wds-modelling-optimization-course/blob/main/notebooks/scp_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sequential convex programming

This notebook applies sequential convex programming (SCP) to a small nonconvex problem with two variables. We cover:
- Building a convex approximation of a nonconvex function
- Solving the convex subproblems with CVXPY
- Using a trust region to accept or reject each step

## Setup
Run the cell below first to import the packages used in this notebook. These packages come preinstalled in Google Colab. To run the notebook locally, install the repository first (see the README).

In [ ]:
import cvxpy as cp
import matplotlib.pyplot as plt
import matplotlib_inline
import numpy as np

# Matplotlib parameters
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300
matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

plt.rc("font", size=10)
plt.rc("axes", titlesize=10)
plt.rc("axes", labelsize=11)
plt.rc("xtick", labelsize=10)
plt.rc("ytick", labelsize=10)
plt.rc("legend", fontsize=10)

## Problem formulation
We consider the nonconvex optimization problem

$$
\begin{aligned}
\min_{x_1,x_2} \quad & f(x_1,x_2) = f_1(x_1,x_2)^2 + f_2(x_1,x_2)^2 \\
\text{subject to} \quad & -5 \leq x_1 \leq 15, \\
& -5 \leq x_2 \leq 15,
\end{aligned}
$$

where

$$
\begin{aligned}
f_1(x_1,x_2) &= x_2 + a_1 x_1^2 + a_2 x_1 + a_3, \\
f_2(x_1,x_2) &= \sqrt{a_4\left(1 + (1-a_5)\cos x_1\right)},
\end{aligned}
$$

and $a_1 = -5.1/(4\pi^2)$, $a_2 = 5/\pi$, $a_3 = -6$, $a_4 = 10$ and $a_5 = 1/(8\pi)$. This is the Branin function, a standard test function with several minima.

Define the parameters and functions.

In [ ]:
# parameters
a1 = -5.1 / (4 * np.pi**2)
a2 = 5 / np.pi
a3 = -6.0
a4 = 10.0
a5 = 1 / (8 * np.pi)


def f1(x1, x2):
    return x2 + a1 * x1**2 + a2 * x1 + a3


def f2(x1, x2):
    arg = np.maximum(1 + (1 - a5) * np.cos(x1), 0.0)  # clip to avoid sqrt of negative values
    return np.sqrt(a4) * np.sqrt(arg)


def f(x1, x2):
    return f1(x1, x2) ** 2 + f2(x1, x2) ** 2

Plot the objective function $f$ over the feasible region.

In [ ]:
x = np.linspace(-5, 15, 200)
y = np.linspace(-5, 15, 200)
X, Y = np.meshgrid(x, y)
Z = f(X, Y)

fig = plt.figure(figsize=(10, 4))

ax = fig.add_subplot(1, 2, 1, projection="3d")
ax.plot_surface(X, Y, Z, cmap="viridis", linewidth=0, antialiased=False, rasterized=True)
ax.set_xlim(-5, 15)
ax.set_ylim(-5, 15)
ax.set_zlim(0, np.max(Z))
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.set_zlabel("$f(x_1,x_2)$")
ax.set_title("Objective surface")

ax2 = fig.add_subplot(1, 2, 2)
cs = ax2.contour(X, Y, Z, levels=80, linewidths=0.5)
ax2.set_xlabel("$x_1$")
ax2.set_ylabel("$x_2$")
ax2.set_title("Objective contours")
fig.colorbar(cs, ax=ax2)

fig.tight_layout()
plt.show()

## Convex approximation
Since $f$ is nonconvex, we cannot solve this problem directly with CVXPY. Instead, SCP starts from an initial point and repeatedly solves a convex approximation of the problem around the current point $(x_1^k, x_2^k)$.

We start from $(x_1^0, x_2^0) = (12, 14.5)$.

In [ ]:
x1_init, x2_init = 12.0, 14.5

print(f"Initial objective value: {f(x1_init, x2_init):.4f}")

We build the convex approximation $\hat{f}$ by linearizing $f_1$ and $f_2$ around $(x_1^k, x_2^k)$:

$$
\hat{f}_i(x_1,x_2) = f_i(x_1^k,x_2^k) + \frac{\partial f_i}{\partial x_1}(x_1^k,x_2^k)\,(x_1-x_1^k) + \frac{\partial f_i}{\partial x_2}(x_1^k,x_2^k)\,(x_2-x_2^k), \quad i = 1, 2,
$$

and setting $\hat{f} = \hat{f}_1^2 + \hat{f}_2^2$. Since $\hat{f}_1$ and $\hat{f}_2$ are affine, $\hat{f}$ is convex: the square function is convex, and convexity is preserved under composition with an affine function. Also, $\hat{f}(x_1^k,x_2^k) = f(x_1^k,x_2^k)$.

The partial derivatives are

$$
\frac{\partial f_1}{\partial x_1} = 2a_1x_1 + a_2, \quad
\frac{\partial f_1}{\partial x_2} = 1, \quad
\frac{\partial f_2}{\partial x_1} = -\frac{\sqrt{a_4}\,(1-a_5)\sin x_1}{2\sqrt{1 + (1-a_5)\cos x_1}}, \quad
\frac{\partial f_2}{\partial x_2} = 0.
$$

In [ ]:
# partial derivatives
def df1_dx1(x1, x2):
    return 2 * a1 * x1 + a2


def df1_dx2(x1, x2):
    return 1.0


def df2_dx1(x1, x2):
    denom = np.sqrt(np.maximum(1 + (1 - a5) * np.cos(x1), 1e-12))  # avoid division by zero
    return -np.sqrt(a4) * (1 - a5) * np.sin(x1) / (2 * denom)


def df2_dx2(x1, x2):
    return 0.0


# linear approximations around the current point (x1k, x2k)
def f1hat(x1, x2):
    return f1(x1k, x2k) + df1_dx1(x1k, x2k) * (x1 - x1k) + df1_dx2(x1k, x2k) * (x2 - x2k)


def f2hat(x1, x2):
    return f2(x1k, x2k) + df2_dx1(x1k, x2k) * (x1 - x1k) + df2_dx2(x1k, x2k) * (x2 - x2k)


# convex approximation of f
def fhat(x1, x2):
    return f1hat(x1, x2) ** 2 + f2hat(x1, x2) ** 2

## SCP algorithm
At each iteration $k$, we solve the convex subproblem

$$
\begin{aligned}
\min_{x_1,x_2} \quad & \hat{f}(x_1,x_2) \\
\text{subject to} \quad & -5 \leq x_1 \leq 15, \\
& -5 \leq x_2 \leq 15, \\
& \lVert x - x^k \rVert_\infty \leq \Delta^k.
\end{aligned}
$$

The last constraint is a *trust region*. It keeps the new point close to $x^k$, where $\hat{f}$ is a good approximation of $f$. Since $\lVert x \rVert_\infty = \max(|x_1|, |x_2|)$, it is equivalent to

$$
|x_1 - x_1^k| \leq \Delta^k, \quad |x_2 - x_2^k| \leq \Delta^k.
$$

Let $\tilde{x}$ be the subproblem solution. We compare the predicted improvement $\hat{\delta} = f(x^k) - \hat{f}(\tilde{x})$ with the actual improvement $\delta = f(x^k) - f(\tilde{x})$:
- If $\delta / \hat{\delta} \geq 0.01$, accept the step ($x^{k+1} = \tilde{x}$) and enlarge the trust region ($\Delta^{k+1} = 1.1\Delta^k$).
- Otherwise, reject the step ($x^{k+1} = x^k$) and shrink the trust region ($\Delta^{k+1} = 0.5\Delta^k$).

We start with $\Delta^0 = 10$. The algorithm stops when the relative improvement $\delta / f(x^k)$ of an accepted step drops below $10^{-3}$, when $\Delta^k$ drops below $10^{-4}$, or after 100 iterations.

In [ ]:
# initial point and trust region size
x1k, x2k = x1_init, x2_init
fk = f(x1k, x2k)
Delta = 10.0
rel_impr = np.inf  # relative improvement of the last accepted step

# history of accepted points and objective values
x_hist = [[x1k, x2k]]
f_hist = [fk]

for k in range(100):
    # define variables
    x1 = cp.Variable()
    x2 = cp.Variable()

    # objective function
    objective = cp.Minimize(fhat(x1, x2))

    # constraints
    constraints = [
        x1 >= -5, x1 <= 15,  # bounds
        x2 >= -5, x2 <= 15,
        cp.abs(x1 - x1k) <= Delta,  # trust region
        cp.abs(x2 - x2k) <= Delta,
    ]

    # solve convex subproblem
    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.CLARABEL)

    # predicted and actual improvements
    predicted_impr = fk - prob.value
    actual_impr = fk - f(x1.value, x2.value)

    if actual_impr / predicted_impr >= 0.01:
        # accept step and enlarge trust region
        rel_impr = actual_impr / fk
        x1k, x2k = x1.value.item(), x2.value.item()
        fk = f(x1k, x2k)
        x_hist.append([x1k, x2k])
        f_hist.append(fk)
        Delta *= 1.1
    else:
        # reject step and shrink trust region
        Delta *= 0.5

    # stopping criteria
    if rel_impr <= 1e-3 or Delta <= 1e-4:
        print(f"Stopped at iteration {k + 1}: relative improvement = {rel_impr:.2e}, Delta = {Delta:.2e}")
        break

x_hist = np.array(x_hist)
f_hist = np.array(f_hist)

print(f"Final point: x1 = {x1k:.4f}, x2 = {x2k:.4f}")
print(f"Final objective value: {fk:.4f}")

## Results
The plots show the SCP path on the contours of $f$ (left) and the objective value at each accepted point (right).

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4))

cs = axs[0].contour(X, Y, Z, levels=80, linewidths=0.5)
axs[0].plot(x_hist[:, 0], x_hist[:, 1], "-x", color="red", linewidth=1.5, markersize=6)
axs[0].set_xlabel("$x_1$")
axs[0].set_ylabel("$x_2$")
axs[0].set_title("SCP path")
fig.colorbar(cs, ax=axs[0])

axs[1].plot(f_hist, "-o", color="black", linewidth=1.5, markersize=4)
axs[1].set_xlabel("Accepted step")
axs[1].set_ylabel("$f(x_1,x_2)$")
axs[1].set_title("Objective value")
axs[1].grid(True, linewidth=0.3)

for ax in axs:
    ax.tick_params(axis="both", direction="in", length=4, width=0.6)
    ax.tick_params(right=True, top=True, labelright=False, labeltop=False)
    for axis in ["top", "bottom", "left", "right"]:
        ax.spines[axis].set_linewidth(0.5)

fig.tight_layout()
plt.show()

## Discussion questions
1. Try different initial trust region sizes $\Delta^0$. How does the trust region size affect exploration vs. exploitation?
2. Try different initial points, e.g. $(-4, 0)$, $(0, 0)$ and $(10, 10)$. How sensitive is the final solution to the initial point?
3. Change the acceptance threshold from 0.01 to 0.1. Are steps accepted more or less often? Why?

## References
- Boyd, S. and Vandenberghe, L. (2004). *Convex Optimization*. Cambridge University Press.
- Diamond, S. and Boyd, S. (2016). CVXPY: A Python-embedded modeling language for convex optimization. *Journal of Machine Learning Research*, 17(83), 1–5.